In [ ]:
# Each thread is given work[id] amount of work.
# Find average work per thread and if a thread's
# work is above (average + K), push the extra
# work to a worklist.
# – This is useful for load-balancing.
# – Also called work-donation.
%%cuda
#include <stdio.h>
#include <cuda.h>

#define N		500
#define BLOCKSIZE	64
#define ELEPERTHREAD	20


__device__ const unsigned delta = ELEPERTHREAD / 5;

__global__ void k1(unsigned *nelements, int nblocks) {
		 __shared__ int sum;
		 __shared__ int donation[BLOCKSIZE*ELEPERTHREAD + 1];
		 __shared__ int donation_idx;

		 unsigned id = blockIdx.x * blockDim.x + threadIdx.x;
		 if(threadIdx.x==0) {
					sum = 0;
					donation_idx = 1;
					donation[0] = -1;
		 }
		 __syncthreads();

     if(id<N) {
				  atomicAdd(&sum, nelements[id]);
		 }
		 __syncthreads();

     int avg;
		 if(blockIdx.x == nblocks-1) {
					avg = sum/(N-blockIdx.x*blockDim.x);
		 }
		 else {
					avg = sum/blockDim.x;
		 }

		 if(id<N) {
				if(nelements[id]>avg+delta) {
						int top = atomicAdd(&donation_idx, nelements[id] - avg - delta);
						for(int i=top;i<top+nelements[id] - avg - delta;i++) {
								donation[i] = id;
						}
				}
		 }
		 __syncthreads();

		 while(1) {
					int work_idx = atomicAdd(&donation_idx, -1) - 1;
					int work_id = donation[work_idx*(work_idx>0)];
					if(work_id==-1) {
							 break;
					}
		 }
}

int main() {
	unsigned hnelements[N];
	for (unsigned ii = 0; ii < N; ++ii) {
		hnelements[ii] = rand() % ELEPERTHREAD;
	}

	unsigned *nelements;
	cudaMalloc(&nelements, N * sizeof(unsigned));
	cudaMemcpy(nelements, hnelements, N * sizeof(unsigned), cudaMemcpyHostToDevice);

	unsigned nblocks = (N + BLOCKSIZE - 1) / BLOCKSIZE;
	k1<<<nblocks, BLOCKSIZE>>>(nelements,nblocks);
	cudaDeviceSynchronize();
	//k2<<<1, 1>>>();
	//cudaDeviceSynchronize();

	return 0;
}